## Importações necessárias

In [72]:
import pretty_midi,muspy,miditok,mido, random
from music21 import *

In [73]:
gramatica = {
    'MELODIA': [('COMPASSO', 1.0)],
    'COMPASSO': [('METADE METADE', 1.0)],              # 2 + 2 = 4 tempos, sempre
    'METADE': [
        ('FORTE 1.0 FRACA 1.0', 0.35),                 # semínima, semínima
        ('FORTE 1.0 FRACA 0.5 FRACA 0.5', 0.30),       # semínima, duas colcheias
        ('FORTE 0.5 FRACA 0.5 FORTE 1.0', 0.20),       # duas colcheias, semínima
        ('FORTE 2.0', 0.15),                           # mínima (respiro)
    ],
    'FORTE': [('1', 0.40), ('3', 0.35), ('5', 0.25)],  # notas do acorde no tempo forte
    'FRACA': [('2', 0.30), ('6', 0.25), ('5', 0.20), ('3', 0.15), ('b3', 0.10)],
}

def expandir(simbolo, contexto=None):
    if simbolo not in gramatica:
        return [simbolo]
    regras, pesos = zip(*gramatica[simbolo])
    escolha = random.choices(regras, weights=pesos, k=1)[0]
    resultado = []
    for s in escolha.split():
        resultado.extend(expandir(s, contexto))
    return resultado

In [74]:
# --- Gramáticas harmônicas (mesmo formato da gramática da melodia) ---
classica = {
    'PECA':        [('ANTECEDENTE CONSEQUENTE', 1.0)],
    'ANTECEDENTE': [('I S D', 0.5), ('I vi S D', 0.5)],     # termina em V (pergunta)
    'CONSEQUENTE': [('I S D I', 0.7), ('T S D I', 0.3)],    # termina em I (resposta)
    'T': [('I', 0.6), ('vi', 0.4)],
    'S': [('IV', 0.5), ('ii', 0.4), ('vi IV', 0.1)],
    'D': [('V', 1.0)],
}

rock = {
    'PECA':   [('VERSO VERSO', 0.4), ('VERSO REFRAO', 0.6)],
    'VERSO':  [('I D vi IV', 0.4), ('I bVII IV I', 0.3), ('I IV D IV', 0.3)],
    'REFRAO': [('vi IV I D', 0.5), ('IV I D vi', 0.3), ('bVI bVII I I', 0.2)],
    'D': [('V', 0.6), ('bVII', 0.4)],
}

# --- Grau romano -> (semitons a partir do tom, sufixo do acorde) ---
GRAUS_ACORDE = {
    'I': (0, ''), 'ii': (2, 'm'), 'iii': (4, 'm'), 'IV': (5, ''),
    'V': (7, ''), 'vi': (9, 'm'),
    'bIII': (3, ''), 'bVI': (8, ''), 'bVII': (10, ''),
}

def grau_para_acorde(grau, tom='C'):
    st, sufixo = GRAUS_ACORDE[grau]
    return NOTAS[(NOTAS.index(tom) + st) % 12] + sufixo

def gerar(simbolo, regras):
    if simbolo not in regras:
        return [simbolo]
    opcoes, pesos = zip(*regras[simbolo])
    escolha = random.choices(opcoes, weights=pesos)[0]
    return [t for s in escolha.split() for t in gerar(s, regras)]

# --- Substitui a progressão fixa ---
graus = gerar('PECA', rock)                     # ou classica
progressao = [grau_para_acorde(g, tom='C') for g in graus]
print(graus, progressao)  # ex.: ['I','bVII','IV','I','vi','IV','I','V'] -> ['C','Bb','F','C','Am','F','C','G']

mapa_graus = {ac: graus_do_acorde(ac) for ac in progressao}

['I', 'IV', 'V', 'IV', 'vi', 'IV', 'I', 'bVII'] ['C', 'F', 'G', 'F', 'Am', 'F', 'C', 'Bb']


In [75]:
estilo = 'rock'            # ou 'classica' (use a gramática correspondente)
TEMPOS_POR_ACORDE = 4.0

# Programas General MIDI
PROGRAMAS = {
    'rock':     {'melodia': 30, 'harmonia': 29, 'baixo': 33},  # guitarra distorcida, guitarra overdrive, baixo
    'classica': {'melodia': 40, 'harmonia': 48, 'baixo': 42},  # violino, naipe de cordas, violoncelo
}

def nova_parte(programa, canal=None):
    p = stream.Part()
    inst = instrument.Instrument()
    inst.midiProgram = programa
    if canal is not None:
        inst.midiChannel = canal
    p.insert(0, inst)
    return p

def m21(nome, oitava):
    return nome[0] + nome[1:].replace('b', '-') + str(oitava)   # Eb -> E-4

melodia  = nova_parte(PROGRAMAS[estilo]['melodia'])
harmonia = nova_parte(PROGRAMAS[estilo]['harmonia'])
baixo    = nova_parte(PROGRAMAS[estilo]['baixo'])
melodia.insert(0, meter.TimeSignature('4/4'))
melodia.insert(0, tempo.MetronomeMark(number=120 if estilo == 'rock' else 90))

for acorde in progressao:
    # Melodia (mesma lógica de antes)
    terminais = expandir('MELODIA', contexto={'acorde': acorde})
    usado = 0.0
    for grau, dur in zip(terminais[0::2], terminais[1::2]):
        dur = min(float(dur), TEMPOS_POR_ACORDE - usado)
        if dur <= 0:
            break
        try:
            nome = grau_para_nota(grau, acorde)
        except KeyError:
            nome = grau_para_nota('1', acorde)
        n = note.Note(m21(nome, 4))
        n.quarterLength = dur
        melodia.append(n)
        usado += dur
    if usado < TEMPOS_POR_ACORDE:
        melodia.append(note.Rest(quarterLength=TEMPOS_POR_ACORDE - usado))

    # Harmonia: power chord (1+5) no rock, tríade na clássica
    graus_h = ['1', '5'] if estilo == 'rock' else ['1', '3', '5']
    notas_h = [m21(grau_para_nota(g, acorde), 3) for g in graus_h]
    if estilo == 'rock':
        notas_h.append(m21(grau_para_nota('1', acorde), 4))      # oitava em cima, som mais cheio
        for _ in range(2):
            harmonia.append(chord.Chord(notas_h, quarterLength=2.0))
    else:
        harmonia.append(chord.Chord(notas_h, quarterLength=4.0))

    # Baixo: colcheias na fundamental no rock, nota longa na clássica
    raiz = m21(grau_para_nota('1', acorde), 2)
    if estilo == 'rock':
        for _ in range(8):
            baixo.append(note.Note(raiz, quarterLength=0.5))
    else:
        baixo.append(note.Note(raiz, quarterLength=4.0))

partes = [melodia, harmonia, baixo]

# Bateria (só no rock): canal 10 do MIDI = índice 9
if estilo == 'rock':
    bateria = nova_parte(0, canal=9)
    BUMBO, CAIXA, CHIMBAL = 36, 38, 42
    for _ in progressao:
        for i in range(8):                          # 8 colcheias por compasso
            golpes = [CHIMBAL]
            if i in (0, 4):
                golpes.append(BUMBO)                # tempos 1 e 3
            if i in (2, 6):
                golpes.append(CAIXA)                # tempos 2 e 4
            bateria.append(chord.Chord([note.Note(midi=g) for g in golpes], quarterLength=0.5))
    partes.append(bateria)

stream.Score(partes).write('midi', fp='music_generated.mid')

'music_generated.mid'

In [76]:
s.show('midi')

In [77]:
music_path = './samples/Goodbye Yellow Brick Road Elton John.mid'

show(music_path)

NameError: name 'show' is not defined